In [1]:
import multiprocessing as mp
mp.set_start_method("spawn", force=True)

### automatically refresh the buffer
%load_ext autoreload
%autoreload 2

### solve the auto-complete issue

%config Completer.use_jedi = False
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

### lvl 2 setups (systerm)
import os
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap,LinearSegmentedColormap,BoundaryNorm
import matplotlib.dates as mdates
import geopandas as gpd
from shapely.geometry import Point
from datetime import datetime
from matplotlib.ticker import FormatStrFormatter

## For ENA, if you want SGP, just replace any 'ENA' to 'SGP'

## qc and to pack to annual dataset

In [2]:
vars_keep = [
    "time_offset",
    "optical_depth_instantaneous",
    "effective_radius_instantaneous",
    "lwp",
    "cloudfraction",
    "ir_temp",
    "cloudbasebestestimate",
    "qc_optical_depth_instantaneous",
    "qc_effective_radius_instantaneous",
    "qc_lwp",
    "qc_cloudfraction",
    "qc_ir_temp",
    "qc_cloudbasebestestimate",
]

dss = xr.open_mfdataset(
    "/data/shared_data/ARM_data/ENA/enamfrsrcldod1minC1.c1/enamfrsrcldod1minC1.c1.2024*.nc",
    preprocess=lambda ds: ds[vars_keep],
    combine="by_coords"
)


In [9]:
import xarray as xr

# variables
cod, re = dss["optical_depth_instantaneous"], dss["effective_radius_instantaneous"]
lwp, cf = dss["lwp"], dss["cloudfraction"]
irt, cbh = dss["ir_temp"], dss["cloudbasebestestimate"]

# QC flags
qc_cod, qc_re = dss["qc_optical_depth_instantaneous"], dss["qc_effective_radius_instantaneous"]
qc_lwp, qc_cf = dss["qc_lwp"], dss["qc_cloudfraction"]
qc_irt, qc_cbh = dss["qc_ir_temp"], dss["qc_cloudbasebestestimate"]

# unified mask
mask = ((qc_cod==0) & (qc_re==0) & (qc_lwp==0) & (qc_cf==0) &
        (qc_irt==0) & (qc_cbh==0) &
        (cf>0.9) & (re!=8) & (re>0) &
        (irt>268.15) & (cbh<4000))

# apply mask
cod_new, re_new = cod.where(mask), re.where(mask)
lwp_new, cf_new = lwp.where(mask), cf.where(mask)
irt_new, cbh_new = irt.where(mask), cbh.where(mask)

# new dataset
ds_new = xr.Dataset(
    {"COD": cod_new, "Re": re_new, "LWP": lwp_new, "CF": cf_new,},
    coords={"time": dss["time_offset"]}
)


ds_new = ds_new.sortby("time")
full_time = pd.date_range(ds_new.time.values[0], ds_new.time.values[-1], freq="20s")
ds_new_full = ds_new.reindex(time=full_time)

ds_new_full.to_netcdf('/data/ggong/ARM_monthly/ENA/mfrsrcldod1minC1.c1/mfrsrcldod1minC1.c1.2024.nc')

## average cloud property from 20-s to 2-min resolution

In [10]:
dsm = xr.open_mfdataset('/data/ggong/ARM_monthly/ENA/mfrsrcldod1minC1.c1/*.nc')

In [11]:
import xarray as xr

outdir = "/data/ggong/ARM_monthly/ENA/mfrsrcldod1minC1.c1"

for year in range(2016, 2026):

    t0 = f"{year}-01-01T00:01:00"
    t1 = f"{year+1}-01-01T00:00:59"

    # time slice
    cl_p = dsm.sel(time=slice(t0, t1))

    if cl_p.time.size == 0:
        continue

    # save
    cl_p.to_netcdf(f"{outdir}/_mfrsrcldod2minC1.c1.{year}.nc")

    print(f"{year} saved")

2016 saved
2017 saved
2018 saved
2019 saved
2020 saved
2021 saved
2022 saved
2023 saved
2024 saved
2025 saved


In [12]:
import xarray as xr
import numpy as np
import pandas as pd

indir = "/data/ggong/ARM_monthly/ENA/mfrsrcldod1minC1.c1"

for year in range(2016, 2026):

    print(f"processing {year}...")

    # read file
    ds = xr.open_dataset(f"{indir}/_mfrsrcldod2minC1.c1.{year}.nc")

    # 1) 2-minute average (01–03, 03–05, ...)
    out = ds.resample(time="2min", offset="1min").mean()

    # 2) Shift the time to the center of the even-numbered minute.
    out = out.assign_coords(time=out.time + np.timedelta64(1, "m"))

    # 3) Generate a complete timeline and fill in NaN values.
    t0 = pd.Timestamp(f"{year}-01-01 00:00:00")
    t1 = pd.Timestamp(f"{year}-12-31 23:58:00")

    even_time = xr.DataArray(
        pd.date_range(t0, t1, freq="2min"),
        dims="time",
        name="time"
    )

    out = out.reindex(time=even_time)

    # 4) save
    outfile = f"{indir}/mfrsrcldod_{year}_2min_resample.nc"
    out.to_netcdf(outfile)

    print(f"saved: {outfile}")

processing 2016...
saved: /data/ggong/ARM_monthly/ENA/mfrsrcldod1minC1.c1/mfrsrcldod_2016_2min_resample.nc
processing 2017...
saved: /data/ggong/ARM_monthly/ENA/mfrsrcldod1minC1.c1/mfrsrcldod_2017_2min_resample.nc
processing 2018...
saved: /data/ggong/ARM_monthly/ENA/mfrsrcldod1minC1.c1/mfrsrcldod_2018_2min_resample.nc
processing 2019...
saved: /data/ggong/ARM_monthly/ENA/mfrsrcldod1minC1.c1/mfrsrcldod_2019_2min_resample.nc
processing 2020...
saved: /data/ggong/ARM_monthly/ENA/mfrsrcldod1minC1.c1/mfrsrcldod_2020_2min_resample.nc
processing 2021...
saved: /data/ggong/ARM_monthly/ENA/mfrsrcldod1minC1.c1/mfrsrcldod_2021_2min_resample.nc
processing 2022...
saved: /data/ggong/ARM_monthly/ENA/mfrsrcldod1minC1.c1/mfrsrcldod_2022_2min_resample.nc
processing 2023...
saved: /data/ggong/ARM_monthly/ENA/mfrsrcldod1minC1.c1/mfrsrcldod_2023_2min_resample.nc
processing 2024...
saved: /data/ggong/ARM_monthly/ENA/mfrsrcldod1minC1.c1/mfrsrcldod_2024_2min_resample.nc
processing 2025...
saved: /data/ggong